In [1]:
import pandas as pd
import numpy as np

from sklearn import preprocessing
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

from xailib.data_loaders.dataframe_loader import prepare_dataframe

from xailib.explainers.lime_explainer import LimeXAITabularExplainer
from xailib.explainers.lore_explainer import LoreTabularExplainer
from xailib.explainers.shap_explainer_tab import ShapXAITabularExplainer

from xailib.models.sklearn_classifier_wrapper import sklearn_classifier_wrapper

import pickle
import altair as alt

import os

In [2]:
path = os.getcwd()
print(path)

/Users/eleonoracappuccio/XAI-Lib_WD/rules_VIZ


In [3]:
source_file = '/Users/eleonoracappuccio/XAI-Lib_WD/datasets/titanic_c.csv' 
class_field = 'Survived'
# Load and transform dataset 
df = pd.read_csv(source_file, skipinitialspace=True, na_values='?', keep_default_na=True)

In [4]:
df

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,1,22.0,1,0,7.2500,2
1,1,1,0,38.0,1,0,71.2833,0
2,1,3,0,26.0,0,0,7.9250,2
3,1,1,0,35.0,1,0,53.1000,2
4,0,3,1,35.0,0,0,8.0500,2
...,...,...,...,...,...,...,...,...
886,0,2,1,27.0,0,0,13.0000,2
887,1,1,0,19.0,0,0,30.0000,2
888,0,3,0,28.0,1,2,23.4500,2
889,1,1,1,26.0,0,0,30.0000,0


In [48]:
df['Age'].max()

80.0

In [5]:
df, feature_names, class_values, numeric_columns, rdf, real_feature_names, features_map = prepare_dataframe(df, class_field)

### Learning a Random Forest classfier

We train a RF classifier by using the ```sklearn``` library. We start by splitting the dataset into a train and test subsets. 

In [6]:
test_size = 0.3
random_state = 42
X_train, X_test, Y_train, Y_test = train_test_split(df[feature_names], df[class_field],
                                                        test_size=test_size,
                                                        random_state=random_state,
                                                        stratify=df[class_field])


Then we train the model on the training set. 
Once the model has been learned, we use a wrapper class to get access to the model for ```XAI lib```

In [7]:
bb = RandomForestClassifier(n_estimators=20, random_state=random_state)
bb.fit(X_train.values, Y_train.values)
bbox = sklearn_classifier_wrapper(bb)

Select a new instance to be classfied by the model and print the predicted class.

In [8]:
inst = X_train.iloc[4].values
print('Instance ',inst)
print('True class ',Y_train.iloc[8])
print('Predicted class ',bb.predict(inst.reshape(1, -1)))

Instance  [ 3.    1.   28.    1.    0.   15.85  2.  ]
True class  0
Predicted class  [0]


In [9]:
real_inst = inst
real_inst

array([ 3.  ,  1.  , 28.  ,  1.  ,  0.  , 15.85,  2.  ])

## Explaining the prediction
We use the explanators of ```XAI lib``` to provide an explantion for the classified instance ```inst```.
Every explainer of ```XAI lib``` takes in input the blackbox to be explained with the corresponding feature names, and a configuration object to initialize the explainer.

### SHAP explainer

In [10]:
explainer = ShapXAITabularExplainer(bbox, feature_names)
config = {'explainer' : 'tree', 'X_train' : X_train.iloc[0:100].values}
explainer.fit(config)

In [11]:
exp = explainer.explain(inst)

In [12]:
exp.plot_features_importance()

alt.VConcatChart(...)

## Learning a different model

### Learning a Logistic Regressor

We train a Logistic Regression by using the ```sklearn``` library. We transform the dataset by using a ```Scaler``` to normalize all the attributes.

In [13]:
scaler = preprocessing.StandardScaler().fit(X_train)
X_scaled = scaler.transform(X_train)

bb = LogisticRegression(C=1, penalty='l2')
bb.fit(X_scaled, Y_train.values)
# pass the model to the wrapper to use it in the XAI lib
bbox = sklearn_classifier_wrapper(bb)

In [14]:
# select a record to explain
inst = X_scaled[182]
print('Instance ',inst)
print('Predicted class ',bb.predict(inst.reshape(1, -1)))

Instance  [-0.3847338   0.73108328 -0.12674551 -0.47468233 -0.4486645  -0.64695437
  0.60029041]
Predicted class  [0]


In [15]:
X_scaled

array([[-1.58920191,  0.73108328, -0.81609293, ..., -0.4486645 ,
         0.46573831,  0.60029041],
       [ 0.81973432,  0.73108328, -0.12674551, ..., -0.4486645 ,
        -0.47826932,  0.60029041],
       [ 0.81973432, -1.36783323, -0.12674551, ..., -0.4486645 ,
        -0.48184838, -0.64217113],
       ...,
       [ 0.81973432, -1.36783323,  1.40513767, ...,  3.11965021,
         0.07336222,  0.60029041],
       [-1.58920191,  0.73108328,  1.32854351, ..., -0.4486645 ,
         0.15980021,  0.60029041],
       [-1.58920191, -1.36783323, -0.81609293, ...,  1.93021197,
        -0.09619664,  0.60029041]])

## Explaining the prediction
We use the same explainators as for the previous model. In this case, a few adjustments are necessary for the initialization of the explanators. For example, SHAP needs a specific configuration for the linear model we are using.

## LIME tabular explainer

In [16]:
limeExplainer = LimeXAITabularExplainer(bbox)
config = {'feature_selection': 'lasso_path'}
limeExplainer.fit(df, class_field, config)
lime_exp = limeExplainer.explain(inst)
print(lime_exp.exp.as_list())# è una lista di tuple

[('Age', -0.08547721643459469), ('Fare', 0.07493915944602668), ('SibSp', 0.011056834467599275), ('Sex', -0.007504485702135668), ('Pclass', -0.006863869875185148), ('Embarked', 0.005292514140583476), ('Parch', 0.0015492594313280725)]


In [56]:
lime_feature_imp=lime_exp.exp.as_list()
lime_feature_imp

[('Age', -0.08547721643459469),
 ('Fare', 0.07493915944602668),
 ('SibSp', 0.011056834467599275),
 ('Sex', -0.007504485702135668),
 ('Pclass', -0.006863869875185148),
 ('Embarked', 0.005292514140583476),
 ('Parch', 0.0015492594313280725)]

In [57]:
lime_exp.plot_features_importance()

alt.VConcatChart(...)

### LORE explainer

In [58]:
explainer = LoreTabularExplainer(bbox)
config = {'neigh_type':'geneticp', 'size':1000, 'ocr':0.1, 'ngen':10}
explainer.fit(df, class_field, config)
exp = explainer.explain(inst)
print(exp)

In [59]:
exp.plotRules()

In [60]:
exp.plotCounterfactualRules()

In [61]:
rules =exp.expDict['rule']['premise']

In [62]:
rules

[{'att': 'Fare', 'op': '<=', 'thr': 10.043749809265137, 'is_continuous': True},
 {'att': 'Age', 'op': '>', 'thr': -5.97114884853363, 'is_continuous': True}]

In [63]:
for r in rules:
    print(r['att'])

Fare
Age


In [64]:
df_range=pd.concat({'min':X_train.min(), 'max':X_train.max()},axis=1)

In [65]:
df_range=df_range.reset_index()

In [66]:
df_range

,index,min,max
0,Pclass,1.00,3.0000
1,Sex,0.00,1.0000
2,Age,0.42,80.0000
3,SibSp,0.00,8.0000
4,Parch,0.00,6.0000
5,Fare,0.00,512.3292
6,Embarked,0.00,2.0000


In [93]:
df_rules = pd.DataFrame.from_records(rules)
df_rules

,att,op,thr,is_continuous
0,Fare,<=,10.043750,True
1,Age,>,-5.971149,True


In [94]:
df_viz = df_range.merge(df_rules,how='left',left_on='index',right_on='att')
df_viz = df_viz.drop('att', axis=1)
df_viz

,index,min,max,op,thr,is_continuous
0,Pclass,1.00,3.0000,NaN,NaN,NaN
1,Sex,0.00,1.0000,NaN,NaN,NaN
2,Age,0.42,80.0000,>,-5.971149,True
3,SibSp,0.00,8.0000,NaN,NaN,NaN
4,Parch,0.00,6.0000,NaN,NaN,NaN
5,Fare,0.00,512.3292,<=,10.043750,True
6,Embarked,0.00,2.0000,NaN,NaN,NaN


In [95]:
df_viz['inst'] = real_inst.tolist()
df_viz

,index,min,max,op,thr,is_continuous,inst
0,Pclass,1.00,3.0000,NaN,NaN,NaN,3.00
1,Sex,0.00,1.0000,NaN,NaN,NaN,1.00
2,Age,0.42,80.0000,>,-5.971149,True,28.00
3,SibSp,0.00,8.0000,NaN,NaN,NaN,1.00
4,Parch,0.00,6.0000,NaN,NaN,NaN,0.00
5,Fare,0.00,512.3292,<=,10.043750,True,15.85
6,Embarked,0.00,2.0000,NaN,NaN,NaN,2.00


In [96]:
thr2_list=[]
for i, row in df_viz.iterrows():
    if (row['op']=='>' or row['op']== '>='):
        thr2_list.append(row['max'])
        continue
    if (row['op']=='<' or row['op']== '<='):
        thr2_list.append(row['min'])
        continue
    else:
        thr2_list.append(np.nan)
df_viz['thr2'] = thr2_list
df_viz

,index,min,max,op,thr,is_continuous,inst,thr2
0,Pclass,1.00,3.0000,NaN,NaN,NaN,3.00,NaN
1,Sex,0.00,1.0000,NaN,NaN,NaN,1.00,NaN
2,Age,0.42,80.0000,>,-5.971149,True,28.00,80.0
3,SibSp,0.00,8.0000,NaN,NaN,NaN,1.00,NaN
4,Parch,0.00,6.0000,NaN,NaN,NaN,0.00,NaN
5,Fare,0.00,512.3292,<=,10.043750,True,15.85,0.0
6,Embarked,0.00,2.0000,NaN,NaN,NaN,2.00,NaN


In [97]:
def add_fi_to_df_and_sort(df,values):
    for string, value in values:
        df.loc[df['index'] == string, 'feature_importance'] = value
        df.sort_values(by=['feature_importance'], key=lambda x: abs(x), ascending=False, inplace=True)
    return df
df_fi=add_fi_to_df_and_sort(df_viz,lime_feature_imp)
df_fi

,index,min,max,op,thr,is_continuous,inst,thr2,feature_importance
2,Age,0.42,80.0000,>,-5.971149,True,28.00,80.0,-0.085477
5,Fare,0.00,512.3292,<=,10.043750,True,15.85,0.0,0.074939
3,SibSp,0.00,8.0000,NaN,NaN,NaN,1.00,NaN,0.011057
1,Sex,0.00,1.0000,NaN,NaN,NaN,1.00,NaN,-0.007504
0,Pclass,1.00,3.0000,NaN,NaN,NaN,3.00,NaN,-0.006864
6,Embarked,0.00,2.0000,NaN,NaN,NaN,2.00,NaN,0.005293
4,Parch,0.00,6.0000,NaN,NaN,NaN,0.00,NaN,0.001549


In [36]:
features=df_viz['index'].to_list()
features

['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']

In [43]:
ch_list=[]
for i, row in df_viz.iterrows():
    if row['inst']!=0 or row['is_continuous']==True:
        p=alt.Chart(
            df_viz[df_viz['index']==row['index']]
        ).mark_rule(
            color='red' if row['is_continuous'] == True else'black',
            size=2
        ).encode(
            x=alt.X(
                field='inst',
                type='quantitative',
                title=None,
                scale= alt.Scale(domain=(row['min'],row['max']),clamp=True,nice=False)
            ),
            tooltip=[alt.Tooltip(field='inst',title=row['index'])]
        )
        
        t_min = alt.Chart(
            df_viz[df_viz['index']==row['index']]
        ).mark_text(
            color='black',
            dx=-10,
            align='right',
#             fontWeight='bold'
        ).encode(
            x=alt.X(
                field='min',
                type='quantitative',
                title=None
            ),
            text='min:N'
        )

        t_max = alt.Chart(
            df_viz[df_viz['index']==row['index']]
        ).mark_text(
            color='grey',
            dx=5,
            align='left',
            fontWeight='bold'
        ).encode(
            x=alt.X(
                field='max',
                type='quantitative',
                title=None
            ),
            text='max:N'
        )
            
        b =alt.Chart(
            df_viz[df_viz['index']==row['index']]
        ).mark_bar(
            color='#f4dd4d',size=5
        ).encode(
            x=alt.X(
                field='thr',
                type='quantitative',
                title=None,
            ),
            x2='thr2',
            y=alt.Y(field='index',type='nominal',title=None),

        )
        
        l =alt.Chart(
            df_viz[df_viz['index']==row['index']]
        ).mark_bar(
            color='grey',size=1
        ).encode(
            x=alt.X(
                field='min',
                type='quantitative',
                title=None,
                scale= alt.Scale(domain=(row['min'],row['max']),clamp=True,nice=False)
            ),
            x2='max',
            y=alt.Y(field='index',type='nominal',title=None),
        )
        
        
        comp = alt.layer(l,b,t_min,t_max,p).properties(
            height=10,
            width=100
        )
        ch_list.append(comp)
concat=alt.vconcat(*ch_list,  title=f"Predicted class: {exp.expDict['bb_pred']}")

concat.configure_concat(
    spacing=0
).configure_axis(
    grid=False
).configure_view(
    strokeWidth=0
).configure_axisX(
    disable=True
).configure_axisY(
    domain=False,
    ticks=False,
    labelPadding=50,
    minExtent=300
).configure_title(
    fontWeight='bold', anchor="start"
)

alt.VConcatChart(...)